<a href="https://colab.research.google.com/github/dipanshurdev/ML-Assignments/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipanshurdev/ML-Assignments/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


#I am using a Random Forest Classifier. Since the goal is to rank pages for a CTR fix, I need a model that outputs probabilities so I can sort by them. A Random Forest handles non-linear relationships well (e.g., CTR behaves very differently at Position 1 vs Position 9) and provides easy-to-read Feature Importances so I can prove the model isn't cheating.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


#I am using a GroupShuffleSplit grouped by client_id. This is the most honest split because it ensures the model is evaluated on clients it has never seen before. It prevents the model from simply memorizing client-specific baseline CTRs.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



# 1. Setup and Load Data
import duckdb
import pandas as pd
import os, getpass
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# (Make sure your HF_TOKEN secret is turned on for notebook access!)
hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
hf_token = hf_token or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

duckdb.sql(f"INSTALL httpfs; LOAD httpfs; CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}');")

BASE_URL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

TARGET_MONTH = "2026-03"

# Load a clean dataframe (ignoring rows with 0 impressions to avoid division by zero)
query = f"""
    SELECT
        client_hash_id, content_hash_id,

        gsc_impressions, gsc_avg_position, ga4_sessions,

        gsc_clicks,
        TRY_CAST(gsc_clicks AS FLOAT) / gsc_impressions as actual_ctr,
        -- Week 4 Baseline Score:
        (gsc_impressions * (0.015 - (TRY_CAST(gsc_clicks AS FLOAT) / gsc_impressions))) as baseline_score
    FROM read_parquet('{BASE_URL}/month={TARGET_MONTH}/*.parquet')

    WHERE gsc_data_available IS TRUE AND gsc_impressions >= 1000 AND gsc_avg_position <= 10
"""
df = duckdb.sql(query).df().fillna(0) # Fill NaNs (like missing word_count) with 0

# 2. Define Features (X) and Label (y)
features = ['gsc_impressions', 'gsc_avg_position', 'ga4_sessions']

df['target_label'] = (df['actual_ctr'] < 0.015).astype(int) # 1 if bad CTR, 0 if good

# 3. Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))


train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

# 4. Train the Model
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(train_df[features], train_df['target_label'])

# Get probabilities for the test set (probability of class 1: bad CTR)
test_df['model_probability'] = rf.predict_proba(test_df[features])[:, 1]

# 5. Evaluate Precision@20 (The Showdown!)
# Sort test set by Model Probability
top_20_model = test_df.sort_values('model_probability', ascending=False).head(20)
model_precision = top_20_model['target_label'].mean()

# Sort test set by Week 4 Baseline Score
top_20_baseline = test_df.sort_values('baseline_score', ascending=False).head(20)
baseline_precision = top_20_baseline['target_label'].mean()

# Base rate (random guessing)
base_rate = test_df['target_label'].mean()

print(f"--- Precision@20 Showdown ---")
print(f"Base Rate (Random Guessing): {base_rate:.1%}")
print(f"Week 4 Baseline Rule: {baseline_precision:.1%}")
print(f"Random Forest Model: {model_precision:.1%}")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Precision@20 Showdown ---
Base Rate (Random Guessing): 92.9%
Week 4 Baseline Rule: 100.0%
Random Forest Model: 100.0%


/tmp/ipykernel_1519/1318189054.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['model_probability'] = rf.predict_proba(test_df[features])[:, 1]


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



# 1. Feature Importances
importances = pd.DataFrame({
    'Feature': features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)
print("--- Feature Importances ---")
display(importances)

# 2. Find the biggest False Positives (Model thought it was BAD CTR, but it was actually GOOD CTR)
false_positives = test_df[(test_df['model_probability'] > 0.8) & (test_df['target_label'] == 0)]
print("\n--- Biggest Mistakes (False Positives) ---")
display(false_positives[['gsc_impressions', 'gsc_avg_position', 'actual_ctr', 'model_probability']].head(3))



--- Feature Importances ---


,Feature,Importance
2,ga4_sessions,0.565664
0,gsc_impressions,0.248012
1,gsc_avg_position,0.186323



--- Biggest Mistakes (False Positives) ---


,gsc_impressions,gsc_avg_position,actual_ctr,model_probability
129,1303,2.882579,0.024559,0.992942
619,1903,5.566999,0.022070,0.996807
852,1271,2.894571,0.025177,0.992968


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.